In [2]:
import duckdb
import pandas as pd

con = duckdb.connect('../data/airbnb.db')
print("Connected to DuckDB")

Connected to DuckDB


In [2]:
# Verify listings_final exists
count = con.execute("SELECT COUNT(*) FROM listings_final").fetchone()[0]
print(f"Source data: {count:,} rows in listings_final")

Source data: 96,182 rows in listings_final


## Star Schema Design Decisions

### Why Star Schema?

**Chosen over:** Normalized (3NF) schema  
**Reasoning:**
- Optimized for analytical queries (OLAP)
- Simpler joins for reporting
- Industry standard for data warehousing

### Dimension Table Keys

| Table | Grain | Surrogate Key | Natural Key |
|-------|-------|---------------|-------------|
| dim_host | One row per unique host | host_id (natural) | host_id |
| dim_location | One row per unique neighbourhood | location_id (surrogate) | neighbourhood_cleansed |
| dim_property | One row per (room_type + property_category) combo | property_id (surrogate) | room_type + category |

### Fact Table Keys

| Key | Type | References |
|-----|------|------------|
| listing_id | Natural PK | listings.id |
| host_id | FK | dim_host.host_id |
| location_id | FK (surrogate) | dim_location.location_id |
| property_id | FK (surrogate) | dim_property.property_id |

### Why Some Keys Are Surrogate?

- `location_id`: Neighbourhood names can change; surrogate key is stable
- `property_id`: Composite key (room_type + category) is cumbersome; surrogate simplifies joins

Creating dim_host

In [3]:
# one row per unique host

con.execute("""
    CREATE OR REPLACE TABLE dim_host AS
    SELECT DISTINCT
        host_id,
        host_name,
        host_since,
        host_location,
        host_is_superhost,
        superhost_status,
        host_response_rate,
        host_acceptance_rate,
        calculated_host_listings_count
    FROM listings_final
    WHERE host_id IS NOT NULL
    ORDER BY host_id
""")

host_count = con.execute("SELECT COUNT(*) FROM dim_host").fetchone()[0]
print(f"dim_host created: {host_count:,} unique hosts")

dim_host created: 57,023 unique hosts


Creating dim_location

In [4]:
# one row per unique neighbourhood

con.execute("""
    CREATE OR REPLACE TABLE dim_location AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY neighbourhood) as location_id,
        neighbourhood,
        AVG(latitude) as avg_latitude,
        AVG(longitude) as avg_longitude,
        COUNT(*) as listing_count
    FROM listings_final
    WHERE neighbourhood IS NOT NULL
    GROUP BY neighbourhood
    ORDER BY neighbourhood
""")

loc_count = con.execute("SELECT COUNT(*) FROM dim_location").fetchone()[0]
print(f" dim_location created: {loc_count} neighbourhoods")

print("\n Top 10 neighborhoods by listing count:")
print(con.execute("""
    SELECT neighbourhood, listing_count
    FROM dim_location
    ORDER BY listing_count DESC
    LIMIT 10
""").df().to_string(index=False))

 dim_location created: 33 neighbourhoods

 Top 10 neighborhoods by listing count:
         neighbourhood  listing_count
           Westminster          10713
         Tower Hamlets           7697
               Hackney           6427
Kensington and Chelsea           6417
                Camden           6377
             Southwark           5387
               Lambeth           5211
             Islington           5148
            Wandsworth           4981
Hammersmith and Fulham           4095


Creating dim_property

In [5]:
# one row per unique (room_type + property_category) combination

con.execute("""
    CREATE OR REPLACE TABLE dim_property AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY room_type, property_category) as property_id,
        room_type,
        property_category,
        COUNT(*) as listing_count
    FROM listings_final
    WHERE room_type IS NOT NULL
    GROUP BY room_type, property_category
    ORDER BY room_type, property_category
""")

prop_count = con.execute("SELECT COUNT(*) FROM dim_property").fetchone()[0]
print(f" dim_property created: {prop_count} unique combinations")

print("\n Property combinations:")
print(con.execute("""
    SELECT room_type, property_category, listing_count
    FROM dim_property
    ORDER BY listing_count DESC
    LIMIT 15
""").df().to_string(index=False))

 dim_property created: 29 unique combinations

 Property combinations:
      room_type property_category  listing_count
Entire home/apt             Other          39770
   Private room             Other          15231
   Private room             House          13596
Entire home/apt         Apartment          10652
Entire home/apt             House          10380
   Private room         Apartment           3659
   Private room             Hotel           1035
   Private room        Guesthouse            486
Entire home/apt              Loft            367
    Shared room             Other            200
   Private room              Loft            171
    Shared room             House            120
     Hotel room         Apartment            103
Entire home/apt             Hotel             76
    Shared room            Hostel             59


Creating facts_listings

In [6]:
# the central fact table

con.execute("""
    CREATE OR REPLACE TABLE fact_listings AS
    SELECT
        l.id as listing_id,
        l.host_id,
        loc.location_id,
        prop.property_id,
        l.price,
        l.price_flag,
        l.minimum_nights,
        l.maximum_nights,
        l.availability_365,
        l.number_of_reviews,
        l.number_of_reviews_ltm,
        l.reviews_per_month,
        l.review_scores_rating,
        l.review_scores_cleanliness,
        l.review_scores_location,
        l.review_scores_communication,
        l.review_scores_value,
        l.first_review,
        l.last_review,
        l.is_active,
        l.instant_bookable
    FROM listings_final l
    LEFT JOIN dim_location loc ON l.neighbourhood = loc.neighbourhood
    LEFT JOIN dim_property prop ON l.room_type = prop.room_type 
                                AND COALESCE(l.property_category, 'Other') = prop.property_category
""")

fact_count = con.execute("SELECT COUNT(*) FROM fact_listings").fetchone()[0]
print(f" fact_listings created: {fact_count:,} rows")

 fact_listings created: 96,182 rows


In [8]:
# Test join: average price by neighborhood

test_query = """
SELECT 
    loc.neighbourhood,
    COUNT(f.listing_id) as listing_count,
    ROUND(AVG(f.price), 2) as avg_price,
    ROUND(MIN(f.price), 2) as min_price,
    ROUND(MAX(f.price), 2) as max_price
FROM fact_listings f
JOIN dim_location loc ON f.location_id = loc.location_id
WHERE f.price IS NOT NULL AND f.price > 0
GROUP BY loc.neighbourhood
ORDER BY avg_price DESC
LIMIT 10
"""

print(" TEST QUERY: Top 10 most expensive neighborhoods:")
print(con.execute(test_query).df().to_string(index=False))


 TEST QUERY: Top 10 most expensive neighborhoods:
         neighbourhood  listing_count  avg_price  min_price  max_price
Kensington and Chelsea           4770     303.76       30.0     5000.0
           Westminster           8057     299.38       19.0     5000.0
        City of London            421     268.85       38.0     5000.0
                Camden           4320     202.76       12.0     5000.0
Hammersmith and Fulham           2659     192.57       25.0     5000.0
  Richmond upon Thames            861     190.63       25.0     2800.0
            Wandsworth           3097     180.45       22.0     2786.0
             Islington           2863     173.14       26.0     5000.0
               Lambeth           3096     159.49       12.0     5000.0
             Southwark           3222     157.95       19.0     4010.0


In [9]:
# Export schema as SQL files

import os
os.makedirs('../sql/schema', exist_ok=True)

# dim_host schema
with open('../sql/schema/01_dim_host.sql', 'w') as f:
    f.write("""-- dim_host: Host dimension table
-- One row per unique host
-- Grain: host_id

CREATE TABLE dim_host (
    host_id BIGINT PRIMARY KEY,
    host_name VARCHAR,
    host_since DATE,
    host_location VARCHAR,
    host_is_superhost BOOLEAN,
    superhost_status VARCHAR,  -- 't', 'f', 'unknown'
    host_response_rate INTEGER,  -- 0-100
    host_acceptance_rate INTEGER,  -- 0-100
    calculated_host_listings_count BIGINT
);
""")

# dim_location schema
with open('../sql/schema/02_dim_location.sql', 'w') as f:
    f.write("""-- dim_location: Location dimension table
-- One row per unique neighbourhood (37 for London)
-- Grain: neighbourhood_cleansed
-- Surrogate key: location_id

CREATE TABLE dim_location (
    location_id INTEGER PRIMARY KEY,
    neighbourhood VARCHAR,
    avg_latitude DOUBLE,
    avg_longitude DOUBLE,
    listing_count BIGINT
);
""")

# dim_property schema (no accommodates)
with open('../sql/schema/03_dim_property.sql', 'w') as f:
    f.write("""-- dim_property: Property dimension table
-- One row per unique (room_type, property_category) combination
-- Grain: room_type + property_category
-- Surrogate key: property_id

CREATE TABLE dim_property (
    property_id INTEGER PRIMARY KEY,
    room_type VARCHAR,
    property_category VARCHAR,
    listing_count BIGINT
);
""")

# fact_listings schema
with open('../sql/schema/04_fact_listings.sql', 'w') as f:
    f.write("""-- fact_listings: Central fact table
-- One row per listing
-- Grain: listing_id
-- Foreign keys: host_id, location_id, property_id

CREATE TABLE fact_listings (
    listing_id BIGINT PRIMARY KEY,
    host_id BIGINT REFERENCES dim_host(host_id),
    location_id INTEGER REFERENCES dim_location(location_id),
    property_id INTEGER REFERENCES dim_property(property_id),
    price DECIMAL,
    price_flag VARCHAR,
    minimum_nights INTEGER,
    maximum_nights INTEGER,
    availability_365 INTEGER,
    number_of_reviews INTEGER,
    number_of_reviews_ltm INTEGER,
    reviews_per_month DOUBLE,
    review_scores_rating DOUBLE,
    review_scores_cleanliness DOUBLE,
    review_scores_location DOUBLE,
    review_scores_communication DOUBLE,
    review_scores_value DOUBLE,
    first_review DATE,
    last_review DATE,
    is_active BOOLEAN,
    instant_bookable BOOLEAN
);
""")

print(" SQL schema files created in sql/schema/")


 SQL schema files created in sql/schema/


### Data enrichment process

Enriching fact_listings with review data

In [10]:
con.execute("""
    CREATE OR REPLACE TABLE fact_listings_enriched AS
    SELECT 
        f.*,
        review_stats.total_reviews,
        review_stats.avg_comments_length,
        review_stats.latest_review_date
    FROM fact_listings f
    LEFT JOIN (
        SELECT 
            listing_id,
            COUNT(*) as total_reviews,
            AVG(LENGTH(comments)) as avg_comments_length,
            MAX(date) as latest_review_date
        FROM reviews
        WHERE comments IS NOT NULL
        GROUP BY listing_id
    ) review_stats ON f.listing_id = review_stats.listing_id
""")

enriched_count = con.execute("SELECT COUNT(*) FROM fact_listings_enriched").fetchone()[0]
print(f" fact_listings_enriched: {enriched_count:,} rows with review data")

 fact_listings_enriched: 96,182 rows with review data


Adding host tenure calculation

In [11]:
con.execute("""
    ALTER TABLE dim_host ADD COLUMN host_tenure_years INTEGER;
    UPDATE dim_host 
    SET host_tenure_years = CASE 
        WHEN host_since IS NOT NULL 
        THEN DATE_DIFF('day', host_since, CURRENT_DATE) / 365
        ELSE NULL
    END;
""")

print(" Added host_tenure_years to dim_host")
print(con.execute("""
    SELECT 
        MIN(host_tenure_years) as min_years,
        AVG(host_tenure_years) as avg_years,
        MAX(host_tenure_years) as max_years
    FROM dim_host WHERE host_tenure_years IS NOT NULL
""").df().to_string(index=False))

 Added host_tenure_years to dim_host
 min_years  avg_years  max_years
         2   9.312445         18


### Running analytic queries

01 - Price by neighbourhood

In [12]:
with open('../sql/analytics/01_price_by_neighbourhood.sql', 'r') as f:
    query = f.read()

result = con.execute(query).df()
print(result)

             neighbourhood  listing_count  avg_price  median_price
0   Kensington and Chelsea           4770     303.76         220.0
1              Westminster           8057     299.38         211.0
2           City of London            421     268.85         195.0
3                   Camden           4320     202.76         155.0
4   Hammersmith and Fulham           2659     192.57         142.0
5     Richmond upon Thames            861     190.63         132.0
6               Wandsworth           3097     180.45         135.0
7                Islington           2863     173.14         137.0
8                  Lambeth           3096     159.49         117.0
9                Southwark           3222     157.95         121.0
10                 Hackney           3303     155.67         126.0
11                  Merton           1006     153.90         110.0
12           Tower Hamlets           4428     152.67         124.0
13                   Brent           2132     149.66         1

In [14]:
with open('../sql/analytics/02_superhost_analysis.sql', 'r') as f:
    query = f.read()

result = con.execute(query).df()
print(result)


  superhost_status  listing_count  avg_price  avg_rating
0                t          17383     178.82        4.85
1                f          77299     185.00        4.64
2          unknown           1488     183.72        4.62


In [15]:
# Read the SQL file and execute it
with open('../sql/analytics/03_room_type_analysis.sql', 'r') as f:
    query = f.read()

result = con.execute(query).df()
print(result)


          room_type property_category  listing_count  avg_price  avg_rating
0   Entire home/apt             Villa             27     759.30        4.83
1       Shared room        Guesthouse             10     339.70        4.28
2   Entire home/apt             House           6811     324.40        4.73
3        Hotel room         Apartment             79     320.05        4.31
4       Shared room            Hostel             53     268.85        4.43
5   Entire home/apt             Hotel             69     257.59        4.53
6   Entire home/apt         Apartment           8181     213.81        4.69
7   Entire home/apt             Other          26864     213.69        4.63
8   Entire home/apt              Loft            205     206.20        4.79
9        Hotel room             Hotel             24     194.92        4.50
10     Private room             Hotel            715     193.21        4.37
11       Hotel room        Guesthouse             10     138.70        4.61
12      Shar

In [16]:
con.close()
print("Connection closed!")

Connection closed!
